###This Colab notebook installs and runs **OrthoFinder**.
It includes:
- One-click **installation** of OrthoFinder and common dependencies (DIAMOND, MMseqs2, MAFFT, MUSCLE, FastTree, mcl).
- **Data upload** or **Google Drive mount**.
- A configurable **run cell** (choose sequence search, MSA, and tree methods).
- **Result inspection** helpers

> ⚠️ **Colab caveats:** Free sessions can disconnect, RAM is limited (~12–25 GB), and CPUs are modest. Use this for demo examples. For larger analyses, use HPC.


In [ ]:
#Run this cell to mount your Google Drive to /content/drive/MyDrive
from google.colab import drive
drive.mount('/content/drive')

## 1) Install OrthoFinder & dependencies

In [ ]:
# System packages
!sudo apt-get update -qq
!sudo apt-get install -y -qq git build-essential python3-dev

# Sequence search & alignment tools
!sudo apt-get install -y -qq diamond-aligner mmseqs2 mafft muscle fasttree mcl

# Utilities
!sudo apt-get install -y -qq pigz unzip

# Python deps
!pip -q install biopython pandas

# OrthoFinder install from source
!rm -rf OrthoFinder
!git clone -q https://github.com/davidemms/OrthoFinder.git
!cd OrthoFinder && python setup.py install --user

# Ensure the user-local bin is on PATH for this session
import os
user_bin = os.path.expanduser("~/.local/bin")
if user_bin not in os.environ.get("PATH",""):
    os.environ["PATH"] = user_bin + ":" + os.environ.get("PATH","")
print("PATH=", os.environ["PATH"])

## 2) Test to see if dependencies installed correctly

In [ ]:
# verify dependencies
!orthofinder -h | head -n 5 || true
!diamond version || true
!mmseqs --version || true
!mafft --version || true
!muscle -version || true
!fasttree -help | head -n 5 || true
!mcl -h | head -n 5 || true


## 3) Provide your protein FASTA files
You have two options (ONLY DO ONE OF THEM):
- **Option A: Upload** protein FASTA files (one per species) — recommended for small demos.
- **Option B: Mount Google Drive** and point to the folder where you have already uploaded your FASTA files.

Each file should be **amino acid sequences** (not DNA). Extensions like `.fa`, `.faa`, or `.fasta` are fine.


In [ ]:
#OPTION A
# Upload files from your computer (select multiple)
from google.colab import files
import os, shutil, glob

INPUT_DIR = "/content/drive/MyDrive/Data" #This should point to the folder containing your data in Google Drive
os.makedirs(INPUT_DIR, exist_ok=True)

print("Select one or more protein FASTA files (.fa/.faa/.fasta):")
uploaded = files.upload()

# Move uploaded files into INPUT_DIR
for fn in uploaded.keys():
    dst = os.path.join(INPUT_DIR, fn)
    shutil.move(fn, dst)

print("\nFiles now in", INPUT_DIR)
!ls -lah /content/drive/MyDrive/Data || true


In [ ]:
# OPTION B
# Mount Google Drive and set an existing folder path containing protein FASTAs
from google.colab import drive
drive.mount('/content/drive/MyDrive/Data')

# After mounting, set this to your Drive folder path (edit as needed):
DRIVE_INPUT_DIR = "/content/drive/MyDrive/Data"

# If you want to *copy* from Drive into the Colab runtime (faster local access), do:
# (Leave commented out if you want OrthoFinder to read directly from Drive.)
# !mkdir -p /content/orthofinder_input
# !cp -v "$DRIVE_INPUT_DIR"/* /content/orthofinder_input/
# print("Copied files into /content/orthofinder_input")


## 4) Validate input files

In [ ]:

import glob, os

INPUT_DIR = "/content/drive/MyDrive/Data"  # change if you prefer Drive folder
files_found = glob.glob(os.path.join(INPUT_DIR, "*"))
if not files_found:
    raise SystemExit("No files found in " + INPUT_DIR + ". Upload or copy your FASTA files first.")

print("Found files:")
for f in files_found:
    print(" -", os.path.basename(f))

# Quick content check: ensure these look like FASTA (first char '>')
bad = []
for f in files_found:
    try:
        with open(f, 'r') as fh:
            first = fh.read(1)
        if first != '>':
            bad.append(f)
    except Exception as e:
        print("Error reading", f, e)

if bad:
    print("\nWARNING: These files do not look like FASTA (no leading '>'):", "\n".join(bad))
else:
    print("\nAll files appear to be FASTA-like.")


## 5) Configure run options

In [ ]:

# Choose methods. Valid options include:
#   SEARCH: 'diamond', 'blast', or 'mmseqs'
#   MSA:    'msa' (MAFFT/MUSCLE) or 'none'
#   TREE:   'fasttree' (recommended in Colab). IQ-TREE is not installed by default here.
SEARCH_METHOD = 'diamond'   # 'diamond' | 'mmseqs' | 'blast'
MSA_METHOD    = 'msa'       # 'msa' | 'none'
TREE_METHOD   = 'fasttree'  # 'fasttree'

# Threads: set based on available CPUs
import os, subprocess
try:
    nproc = int(subprocess.check_output(['nproc']).decode().strip())
except Exception:
    nproc = 2
THREADS = max(1, nproc - 0)  # use all detected cores
print(f"SEARCH={SEARCH_METHOD}, MSA={MSA_METHOD}, TREE={TREE_METHOD}, THREADS={THREADS}")


## 6) Run OrthoFinder

In [ ]:

import os, time, glob, re, subprocess, shlex, pathlib

INPUT_DIR = "/content/drive/MyDrive/Data"  # adjust if needed

# Build command
cmd = [
    "orthofinder",
    "-f", INPUT_DIR,
    "-S", SEARCH_METHOD,
    "-M", MSA_METHOD,
    "-T", TREE_METHOD,
    "-t", str(THREADS)
]

print("Running:", " ".join(shlex.quote(x) for x in cmd))
start = time.time()
ret = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
end = time.time()

# Save log
log_path = "/content/orthofinder_run.log"
with open(log_path, "w") as fh:
    fh.write(ret.stdout)

print(f"Runtime: {end - start:.1f} sec")
print("\n--- Log head ---")
print("\n".join(ret.stdout.splitlines()[:40]))
print("\n... (full log saved to", log_path, ")")

# Find newest Results_* directory
results_dirs = sorted(glob.glob(os.path.join(INPUT_DIR, "OrthoFinder", "Results_*")))
if results_dirs:
    RESULTS_DIR = results_dirs[-1]
    print("\nLatest results directory:", RESULTS_DIR)
else:
    RESULTS_DIR = None
    print("\nNo Results_* directory found. Check the log for errors.")


## 7) Inspect key outputs

In [ ]:
import os, glob, pandas as pd

if RESULTS_DIR and os.path.isdir(RESULTS_DIR):
    print("Top-level files:")
    !ls -lah "$RESULTS_DIR" | head -n 50

    # Common useful tables (only show if present)
    tables = {
        "Orthogroups.tsv": os.path.join(RESULTS_DIR, "Orthogroups", "Orthogroups.tsv"),
        "Orthogroups_SingleCopyOrthologues.txt": os.path.join(RESULTS_DIR, "Orthogroups", "Orthogroups_SingleCopyOrthologues.txt"),
        "Statistics_Overall.tsv": os.path.join(RESULTS_DIR, "Comparative_Genomics_Statistics", "Statistics_Overall.tsv"),
    }

    for label, path in tables.items():
        if os.path.exists(path):
            print(f"\n### {label}")
            try:
                if path.endswith(".tsv") or path.endswith(".txt"):
                    df = pd.read_csv(path, sep='\t')
                    display(df.head(10))
                else:
                    with open(path, 'r') as fh:
                        for i, line in enumerate(fh):
                            if i>20: break
                            print(line.rstrip())
            except Exception as e:
                print("Could not display", label, ":", e)
        else:
            print(f"{label} not found.")
else:
    print("No results directory to inspect.")
